# Your own problem

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimlab-wustl/Nimbus/blob/main/docs/tutorials/02_your_own_problem.ipynb)

Write a problem in a few lines and solve it, with no code beyond the problem itself. The example: pick a team of
exactly 3 out of 7 people. Some pairs work well together, one trio works especially well, and one person brings a
bonus of their own. The constraint, *exactly three*, becomes moves that exchange a member for a non-member, so every
state the machine visits is a team of three.

In [1]:
# In Google Colab, fetch the repository and install the package; locally, run from the repository.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("Nimbus"):
    !git clone -q https://github.com/aimlab-wustl/Nimbus.git
    !pip install -q -e Nimbus
if os.path.exists("Nimbus"):
    os.chdir("Nimbus")
elif os.path.basename(os.getcwd()) == "tutorials":
    os.chdir("../..")                                   # docs/tutorials -> repository root

## 1. The objective and the constraint

Spin *i* is +1 when person *i* is in the team. The energy is E = −Σ J_k Π s_i over the listed products of spins,
of any order; the constraint block says that exactly 3 of the 7 spins are +1.

In [2]:
import yaml, pathlib

problem = {
    "schema": "nimbus.spinpoly/1",
    "name": "team_of_three",
    "n_spins": 7,
    "spin_convention": "plus_one_selected",
    "energy_convention": "minus_J",
    "const": 0,
    "terms": [
        {"support": [0, 1], "weight": 1.0},     # a good pair
        {"support": [1, 4], "weight": 0.8},
        {"support": [2, 5], "weight": 1.2},
        {"support": [3, 6], "weight": -0.6},    # a pair that does not work well
        {"support": [2, 5, 6], "weight": 1.5},  # a trio: a third-order term
        {"support": [6], "weight": 0.4},        # a person with a bonus of their own
    ],
    "constraints": {"exact_cardinality": [{"spins": list(range(7)), "k": 3}]},
}
folder = pathlib.Path("local_runs/notebook_team"); folder.mkdir(parents=True, exist_ok=True)
(folder / "team.yaml").write_text(yaml.safe_dump(problem, sort_keys=False, default_flow_style=None))
print((folder / "team.yaml").read_text())

schema: nimbus.spinpoly/1
name: team_of_three
n_spins: 7
spin_convention: plus_one_selected
energy_convention: minus_J
const: 0
terms:
- support: [0, 1]
  weight: 1.0
- support: [1, 4]
  weight: 0.8
- support: [2, 5]
  weight: 1.2
- support: [3, 6]
  weight: -0.6
- support: [2, 5, 6]
  weight: 1.5
- support: [6]
  weight: 0.4
constraints:
  exact_cardinality:
  - spins: [0, 1, 2, 3, 4, 5, 6]
    k: 3



## 2. The run configuration

The move library `default` builds the moves of every constraint block: here, exchanges of a selected and an
unselected spin. The temperatures are calibrated automatically.

In [3]:
from nimbus import api

config = {
    "schema": "nimbus.run/1",
    "name": "team_of_three",
    "problem": {"kind": "spinpoly", "path": "team.yaml"},
    "transitions": {"library": "default"},
    "annealing": {"schedule": "fn_endpoints", "params": "auto",
                  "calibration": {"rule": "acceptance", "p_start": 0.8, "p_end": 0.01, "eta": 0.1},
                  "attempts": 400},
    "run": {"seed": 1, "replicas": 8},
    "backend": {"name": "cpu"},
    "output": {"overwrite": True},
}
cfg = api.load_config(config, base_dir=folder)
outcome = api.solve(cfg, output_dir=folder / "run")
print(outcome.status)

ok


## 3. The answer

The best team of every replica, decoded from its spins, and a check against all 35 possible teams.

In [4]:
import itertools
import numpy as np
import pandas as pd

rows = pd.read_parquet(folder / "run" / "results.parquet")
print("lowest energy per replica:", rows["best_feasible_objective"].round(3).tolist())

def energy(team):
    s = np.array([1 if i in team else -1 for i in range(7)])
    return -sum(t["weight"] * np.prod(s[t["support"]]) for t in problem["terms"])

best = min(itertools.combinations(range(7), 3), key=energy)
print("best of all 35 teams:", best, "energy", round(energy(best), 3))

lowest energy per replica: [-5.5, -5.5, -5.5, -5.5, -5.5, -5.5, -5.5, -5.5]
best of all 35 teams: (2, 5, 6) energy -5.5


## Next

* Larger problems, other constraint families (precedence, covering, packing, contiguity) and the OPB format:
  [Solving a new problem](https://aimlab-wustl.github.io/Nimbus/guides/new_problem/).
* A problem with its own move library: [A problem adapter](https://aimlab-wustl.github.io/Nimbus/guides/new_problem/adapter/).
* Moves for a constraint the package does not know: the notebook [Finding moves](https://aimlab-wustl.github.io/Nimbus/tutorials/03_finding_moves/).